# TP1 — Embeddings et séquences

**Cours :** Réseaux de neurones récurrents (RNN) — 20 h — Master 1 SDIA
**Enseignante :** Dr. Noulapeu Ngaffo Armielle — ENSPD, Université de Douala
**Durée conseillée :** 1 h 30 · **Barème :** 10 points

---

## Objectifs

- Construire un vocabulaire et indexer un corpus.
- Padder un lot et construire le masque.
- Manipuler une couche d'embedding et calculer une moyenne masquée.

## Travail demandé

1. Répondre aux 3 questions de la section « À vous ».
2. Rendre le notebook exécuté.

## Mode d'emploi

- Exécuter les cellules dans l'ordre. Tout tourne sur **CPU** (quelques minutes).
- Les cellules **« À vous »** sont évaluées.
- Graines fixées : deux exécutions donnent le même résultat.

In [1]:
%matplotlib inline
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(4112); np.random.seed(4112)
torch.set_num_threads(2)
print('torch', torch.__version__)

torch 2.15.0+cpu


## 1. Un vocabulaire, des identifiants

On construit le vocabulaire d'un petit corpus de critiques, puis on convertit chaque phrase en suite d'entiers.

In [2]:
corpus = [
    "le film est excellent et les acteurs sont formidables",
    "un chef d oeuvre absolu je recommande vivement",
    "histoire captivante et mise en scene superbe",
    "je me suis ennuye du debut a la fin",
    "le scenario est confus et les dialogues sont plats",
    "film mediocre sans interet et trop long",
]
mots = sorted({m for p in corpus for m in p.split()})
vocab = {"<pad>": 0, "<unk>": 1}
for m in mots:
    vocab[m] = len(vocab)
inv = {i: m for m, i in vocab.items()}
print("taille du vocabulaire :", len(vocab))
print("premiers identifiants :", list(vocab.items())[:8])

taille du vocabulaire : 42
premiers identifiants : [('<pad>', 0), ('<unk>', 1), ('a', 2), ('absolu', 3), ('acteurs', 4), ('captivante', 5), ('chef', 6), ('confus', 7)]


## 2. Encoder une phrase

In [3]:
def encode(phrase, max_len=None):
    ids = [vocab.get(m, vocab["<unk>"]) for m in phrase.split()]
    if max_len:
        ids = ids[:max_len] + [0] * max(0, max_len - len(ids))
    return ids

for p in corpus[:3]:
    print(len(encode(p)), encode(p))
print("\nlongueurs :", [len(encode(p)) for p in corpus])

9 [24, 17, 14, 16, 15, 25, 4, 36, 19]
8 [40, 6, 8, 30, 3, 22, 32, 41]
7 [20, 5, 15, 29, 12, 35, 38]

longueurs : [9, 8, 7, 9, 9, 7]


## 3. Padding et masque

Un lot doit être rectangulaire. On complète avec 0, et on construit un **masque** pour que ces positions ne comptent pas.

In [4]:
longueur = max(len(encode(p)) for p in corpus)
lot = torch.tensor([encode(p, longueur) for p in corpus])
masque = (lot != 0).float()
print("forme du lot :", tuple(lot.shape))
print("premiere phrase :", lot[0].tolist())
print("son masque      :", masque[0].tolist())
print("mots reels par phrase :", masque.sum(1).tolist())

forme du lot : (6, 9)
premiere phrase : [24, 17, 14, 16, 15, 25, 4, 36, 19]
son masque      : [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
mots reels par phrase : [9.0, 8.0, 7.0, 9.0, 9.0, 7.0]


## 4. L'embedding : un mot devient un vecteur

In [5]:
dim = 16
emb = nn.Embedding(len(vocab), dim, padding_idx=0)
vecteurs = emb(lot)
print("forme :", tuple(vecteurs.shape), "= (lot, longueur, dimension)")

# le vecteur du mot 'film'
i = vocab["film"]
print("vecteur de 'film' (8 premieres valeurs) :")
print(vecteurs[0, lot[0].tolist().index(i), :8].detach().numpy().round(3))

forme : (6, 9, 16) = (lot, longueur, dimension)
vecteur de 'film' (8 premieres valeurs) :
[-1.05  -1.89  -1.419 -0.935 -0.31  -0.056 -0.376  2.096]


## 5. Moyenne masquée : un premier résumé de phrase

Sans masque, les zéros de padding diluent la moyenne. Comparez.

In [6]:
def moyenne_masquee(v, m):
    m = m.unsqueeze(-1)
    return (v * m).sum(1) / m.sum(1).clamp(min=1)

sans = vecteurs.mean(1)
avec = moyenne_masquee(vecteurs, masque)
print("sans masque, phrase 0 :", sans[0, :4].detach().numpy().round(3))
print("avec masque, phrase 0 :", avec[0, :4].detach().numpy().round(3))
print("norme de l'ecart      :", float((sans - avec).norm()))

sans masque, phrase 0 : [-0.54  -0.452  0.354 -0.153]
avec masque, phrase 0 : [-0.54  -0.452  0.354 -0.153]
norme de l'ecart      : 0.49345678091049194


C:\Users\HP\AppData\Local\Temp\ipykernel_11476\2849331230.py:9: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:821.)
  print("norme de l'ecart      :", float((sans - avec).norm()))


## 6. À vous

**1.** Ajoutez deux phrases au corpus et vérifiez que le vocabulaire grandit.

**2.** Que se passe-t-il si l'on met `padding_idx=0` puis qu'on oublie le masque ? Mesurez l'écart sur une phrase courte.

**3.** Pourquoi réserve-t-on l'indice 0 au padding plutôt qu'au mot le plus fréquent ?

In [7]:
# ---- A VOUS ----
# 1)

# 2)

# 3) reponse :

## 7. Synthèse

| Étape | Résultat |
|---|---|
| tokenisation | liste de mots |
| indexation | liste d'entiers |
| padding | matrice rectangulaire |
| masque | les positions de remplissage ne comptent pas |
| embedding | tenseur (lot, longueur, dimension) |